# UCSC Custom Track Generation (TrunCat predictions)

Converts existing TrunCat prediction CSVs (TOPMed, ClinVar, gnomAD, GREGoR) into a bigBed track
hostable directly from GitHub (raw.githubusercontent.com supports the HTTP byte-range requests UCSC
needs -- no GitHub Pages setup or separate hosting required).

**Pipeline:** CSV predictions -> BED (0-based, half-open, indel-aware) -> bigBed -> hub files

**Requirements (one-time setup, not pip-installable):**
```bash
conda install -c bioconda ucsc-bedtobigbed
```

**Input format expected** (one row per variant, columns from your `predict/*.csv` outputs):
`GENE_ID, hgnc_symbol, CHROM, POS, variantID, escape_probability, predicted_class, predicted_label, threshold_used`

`variantID` is expected in `chrom_pos_ref_alt` format (e.g. `chr10_100286116_G_T`) -- the ref allele is
parsed out of this to get the correct BED end-coordinate for indels, not just SNVs.

## 0. Configuration

In [ ]:
import pandas as pd
import numpy as np
import subprocess
import shutil
from pathlib import Path

# --- Input prediction files (adjust paths as needed) ---
PREDICT_DIR = Path("../predict")  # NMDpredictionmodel/Model/TrunCat/predict/

PREDICTION_FILES = {
    'gnomAD': PREDICT_DIR / "gnomAD_predictions.csv",
    'ClinVar': PREDICT_DIR / "clinvar_predictions.csv",
    'GREGoR': PREDICT_DIR / "gregor_predictions.csv",
    # 'TOPMed': PREDICT_DIR / "topmed_predictions.csv",  # uncomment if/when available
}

MODEL_NAME = "TrunCat"
GENOME_BUILD = "hg38"

# --- Output location ---
OUTPUT_DIR = Path("../track")  # will hold the .bed, .bb, .as, and hub files
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRACK_NAME = "truncat_predictions"
BED_PATH = OUTPUT_DIR / f"{TRACK_NAME}.bed"
BIGBED_PATH = OUTPUT_DIR / f"{TRACK_NAME}.bb"
AS_PATH = OUTPUT_DIR / f"{TRACK_NAME}.as"
CHROM_SIZES_PATH = OUTPUT_DIR / f"{GENOME_BUILD}.chrom.sizes"

# --- Your GitHub repo details, for the hub files at the end ---
GITHUB_USER = "CobanAkdemirLab"           # TODO: confirm
GITHUB_REPO = "NMDpredictionmodel"
GITHUB_BRANCH = "main"
GITHUB_TRACK_PATH = "Model/TrunCat/track"  # path within the repo where files will be committed

print(f"Reading predictions from: {PREDICT_DIR.resolve()}")
print(f"Writing track files to:   {OUTPUT_DIR.resolve()}")
for name, path in PREDICTION_FILES.items():
    print(f"  {name}: {path}  {'\u2713' if path.exists() else '\u2717 NOT FOUND'}")

## 1. Load and combine predictions

In [ ]:
REQUIRED_COLS_BASE = ['GENE_ID', 'hgnc_symbol', 'CHROM', 'variantID',
                       'escape_probability', 'predicted_class', 'predicted_label', 'threshold_used']

def derive_pos_from_variant_id(variant_id):
    """'chr7_127588544_A_T' -> 127588544 (int), or None if the format doesn't match."""
    parts = str(variant_id).split('_')
    if len(parts) != 4:
        return None
    try:
        return int(parts[1])
    except ValueError:
        return None

dfs = []
for cohort, path in PREDICTION_FILES.items():
    if not path.exists():
        print(f"Skipping {cohort} -- file not found at {path}")
        continue
    df_c = pd.read_csv(path)

    missing = [c for c in REQUIRED_COLS_BASE if c not in df_c.columns]
    if missing:
        raise ValueError(f"{cohort} file is missing expected columns: {missing}")

    if 'POS' not in df_c.columns:
        print(f"  {cohort}: no POS column -- deriving it from variantID")
        df_c['POS'] = df_c['variantID'].apply(derive_pos_from_variant_id)
        n_failed = df_c['POS'].isna().sum()
        if n_failed:
            raise ValueError(
                f"{cohort}: could not derive POS from variantID for {n_failed} row(s) -- "
                f"check variantID format matches 'chrom_pos_ref_alt'. Examples:\n"
                f"{df_c.loc[df_c['POS'].isna(), 'variantID'].head(5).to_string(index=False)}"
            )
        df_c['POS'] = df_c['POS'].astype(int)

    df_c['cohort'] = cohort
    dfs.append(df_c)
    print(f"\u2713 {cohort}: {len(df_c)} variants")

df = pd.concat(dfs, ignore_index=True)
print(f"\nTotal variants across all cohorts: {len(df)}")

n_dupe = df.duplicated(subset=['variantID']).sum()
if n_dupe:
    print(f"\nNote: {n_dupe} variantID(s) appear in more than one cohort file "
          f"(e.g. a variant present in both gnomAD and ClinVar) -- these will appear as separate "
          f"track entries, one per cohort, distinguished by the 'cohort' field.")

df.head(3)

In [ ]:
# ============================================================================
# Section 1b — Merge in transcript names from the annotation files
# ============================================================================

TXNAME_FILES = {
    'gnomAD': PREDICT_DIR / "gnomAD_stopgain.df_April152026.csv",
    'ClinVar': PREDICT_DIR / "clinVar_stopgain.df_April152026.csv",
    'GREGoR': PREDICT_DIR / "GREGoR_stopgain.df_May42026.csv",
}

tx_dfs = []
for cohort, path in TXNAME_FILES.items():
    if not path.exists():
        print(f"WARNING: {cohort} annotation file not found at {path} -- transcript info will be missing for this cohort")
        continue
    tx_c = pd.read_csv(path, usecols=lambda c: c in ('variantID', 'txnames'))
    if 'txnames' not in tx_c.columns:
        raise ValueError(f"{cohort} annotation file has no 'txnames' column -- check the actual column name")
    tx_dfs.append(tx_c[['variantID', 'txnames']])
    print(f"\u2713 {cohort}: {len(tx_c)} variant-transcript rows loaded")

tx_df = pd.concat(tx_dfs, ignore_index=True).drop_duplicates(subset='variantID')

# Preview the actual format before we commit to how it's presented in the track
print("\nSample txnames values (checking whether this is single-transcript or multi-transcript per variant):")
print(tx_df['txnames'].head(10).to_string(index=False))

n_multi = tx_df['txnames'].astype(str).str.contains(r'[,;]').sum()
print(f"\n{n_multi} of {len(tx_df)} rows appear to contain more than one transcript (comma/semicolon-separated).")
if n_multi > 0:
    print("NOTE: if these are ALL overlapping transcripts rather than the one transcript minicat actually\n"
          "used for prediction, the track field should probably be labeled 'overlapping transcripts' rather\n"
          "than 'transcript used for prediction' -- confirm which this is before finalizing the .as schema below.")

# Merge onto the main predictions table
df = df.merge(tx_df, on='variantID', how='left')

n_missing_tx = df['txnames'].isna().sum()
if n_missing_tx:
    print(f"\nWARNING: {n_missing_tx} of {len(df)} predicted variants have no matching transcript "
          f"annotation (variantID not found in the annotation files). These will get an empty transcript "
          f"field in the track.")
df['txnames'] = df['txnames'].fillna('NA')

## 2. Parse variantID and convert to BED coordinates (0-based, half-open, indel-aware)

In [ ]:
def parse_variant_id(variant_id):
    """Parses 'chr10_100286116_G_T' -> (chrom, pos, ref, alt).
    Returns (chrom, pos, ref, alt) or (None, None, None, None) if the format doesn't match."""
    parts = str(variant_id).split('_')
    if len(parts) != 4:
        return None, None, None, None
    chrom, pos, ref, alt = parts
    try:
        pos = int(pos)
    except ValueError:
        return None, None, None, None
    return chrom, pos, ref, alt

parsed = df['variantID'].apply(parse_variant_id)
df['vid_chrom'] = parsed.apply(lambda x: x[0])
df['vid_pos'] = parsed.apply(lambda x: x[1])
df['ref'] = parsed.apply(lambda x: x[2])
df['alt'] = parsed.apply(lambda x: x[3])

n_unparsed = df['ref'].isna().sum()
if n_unparsed:
    print(f"WARNING: {n_unparsed} variantID(s) could not be parsed as 'chrom_pos_ref_alt' -- "
          f"falling back to a 1bp SNV-style window using CHROM/POS for these. Inspect if this count is large:")
    print(df.loc[df['ref'].isna(), 'variantID'].head(10).to_string(index=False))

# Sanity check: CHROM/POS columns should agree with what's parsed out of variantID
mismatch = df[df['vid_chrom'].notna() & ((df['vid_chrom'] != df['CHROM']) | (df['vid_pos'] != df['POS']))]
if len(mismatch):
    print(f"\nWARNING: {len(mismatch)} row(s) where CHROM/POS disagree with the parsed variantID -- "
          f"using CHROM/POS as the source of truth for these:")
    print(mismatch[['variantID', 'CHROM', 'POS', 'vid_chrom', 'vid_pos']].head(10).to_string(index=False))

# --- Build BED coordinates ---
# VCF-style POS is 1-based. BED is 0-based, half-open: start = POS-1, end = start + len(ref).
# Falls back to a 1bp window (SNV assumption) when ref couldn't be parsed.
df['ref_len'] = df['ref'].apply(lambda r: len(r) if isinstance(r, str) and r.isalpha() else 1)
df['bed_start'] = df['POS'].astype(int) - 1
df['bed_end'] = df['bed_start'] + df['ref_len']

print(f"\n\u2713 BED coordinates built for {len(df)} variants")
df[['variantID', 'CHROM', 'POS', 'ref', 'alt', 'bed_start', 'bed_end']].head(5)

## 3. Build the BED table (BED6 + custom extra fields)

In [ ]:
bed_df = pd.DataFrame({
    'chrom': df['CHROM'],
    'chromStart': df['bed_start'],
    'chromEnd': df['bed_end'],
    'name': df['variantID'],
    # BED score field is an integer 0-1000; scale escape_probability (0-1 float) accordingly
    'score': (df['escape_probability'] * 1000).round().clip(0, 1000).astype(int),
    'strand': '.',  # not available at the variant level in this prediction table
    # --- extra (non-standard) fields, defined in the .as schema below ---
    'geneId': df['GENE_ID'],
    'hgncSymbol': df['hgnc_symbol'],
    'transcriptId': df['txnames'],
    'escapeProbability': df['escape_probability'].round(6),
    'predictedLabel': df['predicted_label'],
    'thresholdUsed': df['threshold_used'].round(6),
    'cohort': df['cohort'],
    'model': MODEL_NAME,
})

# bedToBigBed requires the file sorted by chrom, then chromStart
bed_df = bed_df.sort_values(['chrom', 'chromStart']).reset_index(drop=True)

print(f"BED table: {len(bed_df)} rows, {bed_df.shape[1]} columns")
bed_df.head(3)

In [ ]:
bed_df.to_csv(BED_PATH, sep='\t', header=False, index=False)
print(f"\u2713 Saved: {BED_PATH}  ({len(bed_df)} entries)")

## 4. Write the autoSql schema (.as) -- tells UCSC what the extra columns mean

In [ ]:
AS_SCHEMA = f'''table {TRACK_NAME}
"{MODEL_NAME} NMD-escape predictions"
(
string  chrom;          "Chromosome"
uint    chromStart;     "Start position (0-based)"
uint    chromEnd;       "End position"
string  name;           "Variant ID (chrom_pos_ref_alt)"
uint    score;          "Escape probability x1000, for track shading (0-1000)"
char[1] strand;         "Strand (not available at variant level; '.')"
string  geneId;         "Ensembl gene ID"
string  hgncSymbol;     "HGNC gene symbol"
string  transcriptId;   "Transcript(s) used for prediction"
float   escapeProbability; "Raw predicted NMD-escape probability (0-1)"
string  predictedLabel; "Predicted class label (NMD / escape)"
float   thresholdUsed;  "Decision threshold used to assign predictedLabel"
string  cohort;         "Source cohort (gnomAD / ClinVar / GREGoR / TOPMed)"
string  model;          "Model used to generate this prediction"
)
'''

with open(AS_PATH, 'w') as f:
    f.write(AS_SCHEMA)
print(f"\u2713 Saved: {AS_PATH}")
print(AS_SCHEMA)

## 5. hg38 chrom.sizes (embedded -- primary assembly only)

Hardcoded rather than downloaded, since this notebook shouldn't need network access just to get a
static reference table. If any of your variants fall on an alt/patch/scaffold contig not listed here,
`bedToBigBed` will error out on that row rather than silently dropping it -- see Section 6.

In [ ]:
HG38_CHROM_SIZES = {
    'chr1': 248956422, 'chr2': 242193529, 'chr3': 198295559, 'chr4': 190214555,
    'chr5': 181538259, 'chr6': 170805979, 'chr7': 159345973, 'chr8': 145138636,
    'chr9': 138394717, 'chr10': 133797422, 'chr11': 135086622, 'chr12': 133275309,
    'chr13': 114364328, 'chr14': 107043718, 'chr15': 101991189, 'chr16': 90338345,
    'chr17': 83257441, 'chr18': 80373285, 'chr19': 58617616, 'chr20': 64444167,
    'chr21': 46709983, 'chr22': 50818468, 'chrX': 156040895, 'chrY': 57227415,
    'chrM': 16569,
}

with open(CHROM_SIZES_PATH, 'w') as f:
    for chrom, size in HG38_CHROM_SIZES.items():
        f.write(f"{chrom}\t{size}\n")
print(f"\u2713 Saved: {CHROM_SIZES_PATH}")

# Flag any variants on contigs not in this primary-assembly list, before bedToBigBed does
unknown_chroms = sorted(set(bed_df['chrom']) - set(HG38_CHROM_SIZES.keys()))
if unknown_chroms:
    n_affected = bed_df['chrom'].isin(unknown_chroms).sum()
    print(f"\nWARNING: {n_affected} variant(s) are on contigs not in the primary-assembly chrom.sizes: "
          f"{unknown_chroms}")
    print("These will fail bedToBigBed conversion. Either exclude them or add their sizes to "
          "HG38_CHROM_SIZES above (full list: https://hgdownload.soe.ucsc.edu/goldenPath/hg38/bigZips/hg38.chrom.sizes).")
else:
    print("\u2713 All variants fall on primary-assembly chromosomes.")

## 6. Convert to bigBed

Requires `bedToBigBed` on PATH. If this cell errors with "command not found":
```bash
conda install -c bioconda ucsc-bedtobigbed
```

In [ ]:
if shutil.which('bedToBigBed') is None:
    raise EnvironmentError(
        "bedToBigBed not found on PATH. Install it with:\n"
        "  conda install -c bioconda ucsc-bedtobigbed\n"
        "then re-run this cell."
    )

n_extra_fields = bed_df.shape[1] - 6  # fields beyond the standard BED6
cmd = [
    'bedToBigBed',
    f'-as={AS_PATH}',
    f'-type=bed6+{n_extra_fields}',
    str(BED_PATH),
    str(CHROM_SIZES_PATH),
    str(BIGBED_PATH),
]
print("Running:", ' '.join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)

if result.returncode != 0:
    print("STDERR:", result.stderr)
    raise RuntimeError("bedToBigBed failed -- see error above (common causes: unsorted BED, "
                        "a chrom missing from chrom.sizes, or an out-of-bounds coordinate).")

print(f"\u2713 Saved: {BIGBED_PATH}  ({BIGBED_PATH.stat().st_size / 1024:.1f} KB)")

## 7. Hub files (hub.txt, genomes.txt, trackDb.txt)

After running this notebook, commit `{TRACK_NAME}.bb` and these three files to your GitHub repo under
`{GITHUB_TRACK_PATH}/`, then load the hub in UCSC via:
`https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{GITHUB_TRACK_PATH}/hub.txt`

In [ ]:
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_USER}/{GITHUB_REPO}/{GITHUB_BRANCH}/{GITHUB_TRACK_PATH}"

hub_txt = f'''hub {TRACK_NAME}_hub
shortLabel {MODEL_NAME} NMD predictions
longLabel {MODEL_NAME} NMD-escape predictions ({", ".join(PREDICTION_FILES.keys())})
genomesFile genomes.txt
email your_email@example.com
'''

genomes_txt = f'''genome {GENOME_BUILD}
trackDb trackDb.txt
'''

trackdb_txt = f'''track {TRACK_NAME}
bigDataUrl {RAW_BASE}/{TRACK_NAME}.bb
shortLabel {MODEL_NAME} predictions
longLabel {MODEL_NAME} NMD-escape predictions, scored variants only (not genome-wide)
type bigBed 6 +
itemRgb off
visibility pack
'''

for fname, content in [('hub.txt', hub_txt), ('genomes.txt', genomes_txt), ('trackDb.txt', trackdb_txt)]:
    out_path = OUTPUT_DIR / fname
    with open(out_path, 'w') as f:
        f.write(content)
    print(f"\u2713 Saved: {out_path}")

print(f"\nOnce committed to GitHub, load the hub in UCSC at:")
print(f"  https://genome.ucsc.edu/cgi-bin/hgHubConnect?hubUrl={RAW_BASE}/hub.txt")

## Done

Files produced in `OUTPUT_DIR`:
- `{TRACK_NAME}.bed` -- intermediate BED file (not needed after conversion, but kept for inspection)
- `{TRACK_NAME}.bb` -- the bigBed file to commit and host
- `{TRACK_NAME}.as` -- schema describing the extra fields
- `{GENOME_BUILD}.chrom.sizes` -- reference used for the conversion
- `hub.txt`, `genomes.txt`, `trackDb.txt` -- hub definition files

**Next step:** commit `{TRACK_NAME}.bb`, `hub.txt`, `genomes.txt`, and `trackDb.txt` to 
`{GITHUB_TRACK_PATH}/` in your repo (the `.bed`, `.as`, and `.chrom.sizes` files don't need to be
committed -- they're just intermediate artifacts), then test-load the hub URL in UCSC.